<a href="https://colab.research.google.com/github/RiccoFlores/100-Days-Of-ML-Code/blob/master/Proyecto_integrador_M4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Proyecto integrador**
**Construcción y Optimización de un Sistema RAG (Retrieval-Augmented Generation)**

Instrucciones
1. Adquisición de documentos
2. Generación de embeddings y carga en FAISS
3. Aplicación del modelo RAG
4. Investigación sobre mejora de rendimiento en RAGs



# **Desarrollo**

**Objetivo:** Comprender el flujo completo de un sistema RAG desde la adquisición de información hasta la generación de respuestas contextuales utilizando un modelo LLM local y gratuito (Qwen2.5-1.5B-Instruct) junto al motor de indexación vectorial FAISS.

### Pasos para el sistema RAG:
1. **Adquisición:** Carga de documentos fuente en formato PDF, se consideran textos sobre el impacto de la inteligencia artificial en la sociedad.
2. **Chunking:** Segmentación del texto en fragmentos lógicos mediante RecursiveCharacterTextSplitter.
3. **Embedding:** Vectorización semántica con sentence-transformers/all-MiniLM-L6-v2.
4. **Vector Store:** Indexación e indexación en base de datos local con FAISS.
5. **Retrieval & Generation:** Recuperación contextual por similitud de coseno y síntesis estricta con el LLM.

# **Documentos**

Se toman 7 documentos en pdf sobre inteligencia artificial, son textos generales sobre aplicaciones, avances, ventajas, desventajas, etc .


- Cáceres, J. D. (2023). La inteligencia artificial y sus implicaciones en el marketing. Palermo Business Review, (27), 39-55.

- Gonzales, Y., Jacobs Estrada, M. C., y Hercules Castro, C. A. (2024). Las ventajas y desventajas de la aplicación de la inteligencia artificial en las ciencias empresariales. LATAM Revista Latinoamericana de Ciencias Sociales y Humanidades, 5(5), 5442-5451. https://doi.org/10.56712/latam.v5i5.3001

- Carrazco-Peña, K. B., Farias-Moreno, K., del Toro-Equihua, M., Aguilar-Mancilla, Z. C., Trujillo-Magallón, M., Solórzano-Rodríguez, M. A., y Trujillo-Hernández, B. (2022). Componentes de fragilidad, sarcopenia y su asociación con insuficiencia de vitamina D. Estudio transversal analítico. Gaceta Médica de México, 158(6), 353-358. https://doi.org/10.24875/GMM.22000104

- Chiliquinga Amaya, J. A., Arcentales Macias, A. M., y Pereira Salcedo, J. R. (2024). La influencia de la inteligencia artificial en la sociedad actual y en el futuro de las próximas generaciones. Sapiens in Artificial Intelligence, 1(1), 37-47. https://revistasapiensec.com/index.php/Sapiens_in_Artificial_Intelligen/article/view/35

- Serrahima de Bedoya, Á. (2022). Avances y desafíos de la inteligencia artificial (J. Bengoechea Fernández, Dir.) [Trabajo de fin de grado, Universidad Pontificia Comillas]. Repositorio Institucional ICAI. Madrid, España.

- Vadillo Bueno, G. (2020). Futuros de la inteligencia artificial. Revista Digital Universitaria, 21(1), Artículo a3. https://doi.org/10.22201/codeic.16076079e.2020.v21n1.a3

- Rouhiainen, L. (2018). Inteligencia artificial: 101 cosas que debes saber hoy sobre nuestro futuro. Alienta Editorial.

Estos documentos se cargan dentro de Colab, pero también se agregarán dentro de las entregas.


# **Código**

In [ ]:
# 1. INSTALACIÓN DE LIBRERÍAS Y CONFIGURACIÓN DEL ENTORNO

# NOTA: EL MODELO SE CORRE EN EL ENTORNO DE EJECUCIÓN T4 PARA MEJORAR LA VELOCIDAD DE RESPUESTA (Entorno de ejecución-->Cambiar tipo de entorno de ejecución--> T4)

# Aquí se instalan las librerías necesarias para poder ejecutar el sistema, en términos generales importamos herramientas de langchain, lector de pdf, cortador de texto, base de datos y conectores de los modelos
# langchain sirve para unir los textos, prompt y modelo
# transformers para descargar y ejecutar los modelos Hugging Face
# faiss-cpu para almacenar los vectores
# pypdf para leer y extraer loa pdf

!pip install -U langchain langchain-core langchain-community langchain-huggingface transformers sentence-transformers faiss-cpu pypdf

import os
import numpy as np
import pandas as pd

# Componentes de LangChain y Hugging Face
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFacePipeline

# Configuración del LLM
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("\n>> [ÉXITO] Entorno e importaciones listos.")


>> [ÉXITO] Entorno e importaciones listos.


In [ ]:

# 2. PROCESAMIENTO, VECTORIZACIÓN Y CONSTRUCCIÓN DEL PIPELINE


# Paso 1: Subir documentos
# (indicamos que se apunte a la carpeta de Colab, donde está la info que queremos procesar)
PATH_A_TUS_PDFS = "/content/"
print(">> Cargando archivos PDF desde la ruta...")
loader = PyPDFDirectoryLoader(PATH_A_TUS_PDFS)
documentos_cargados = loader.load()
print(f">> Páginas totales procesadas: {len(documentos_cargados)}")

if len(documentos_cargados) == 0:
    raise ValueError("Sube tus PDFs antes de ejecutar esta celda.")


# Paso 2: Generación de Chunks (Segmentación)
#(Cortamos los textos para que la IA pueda leerlos, le damos 1000 caracteres como máximo para cada fragmento, también le pedimos que el final de un fragmento comparta 200 caracteres con el inicio del siguiente con el fin de no perder contexto)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documentos_cargados)
print(f">> Total de chunks generados: {len(chunks)}")


# Paso 3: Carga de Embeddings e Indexación en FAISS
#(Convierte en número (vectores) los fragmentos y los organiza en un mapa, para que al preguntar traiga los 3 fragmentos que se parezcan más a la pregunta)
print(">> Creando embeddings numéricos e indexando en FAISS...")
model_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, model_embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})


# Paso 4: Inicialización del LLM Local (Qwen 2.5)
#(Descarga del modelo de lenguaje de código abierto Qwewn 2.5, el tokenizer traduce las palabras a código numérico y device_map le pide a Colab cargar de forma automática el modelo dentro de la tarjeta gráfica GPU para aumentar la velocidad de respuesta )
print(">> Descargando y configurando LLM local en GPU...")
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

#(Aquí le pedimos un máximo de 512 palabras para evitar respuestas muy largas, la temperatura de 0.1 vuelve almodelo lógico, preciso y estricto, con el fin de evitar alucionaciones)
# Para este ejercicio con el comando return_full_text en False le pido que no me dé los fragmentos de donde genera la respuesta
pipe = pipeline(
    "text-generation", model=model, tokenizer=tokenizer,
    max_new_tokens=512, temperature=0.1, do_sample=True,
    return_full_text=False
)
llm = HuggingFacePipeline(pipeline=pipe)


# Paso 5: Prompt Engineering Estricto y Construcción LCEL
#(Aquí se configura la personalidad del modelo, en este caso se le pide que sea riguroso y en caso de no contar con la respuesta en los PDF lo mencione)
system_prompt = (
    "Eres un asistente analítico avanzado y riguroso. Tu único objetivo es responder la pregunta del usuario "
    "basándote de forma exclusiva y estricta en el contexto proporcionado a continuación.\n"
    "Si la respuesta no se encuentra explícitamente en el contexto, debes responder exactamente: "
    "'No cuento con información suficiente en los documentos proporcionados'.\n\n"
    "Contexto extraído de los PDFs:\n{context}"
)
prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt), ("human", "{input}")
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


#Aquí se define el flujo de los datos: entra la pregunta (RunnablePassthrough)-->Busca en los textos (retriever)-->Los une (format _docs)-->Agrega el contexto y pregunta (promt_template)-->Envía el prompt a LLM Qwen para que genere la respuesta-->Limpia resultado para entregar la respuesta

rag_pipeline = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt_template | llm | StrOutputParser()
)
print(">> [ÉXITO] Pipeline RAG inicializado por completo.")

>> Cargando archivos PDF desde la ruta...
>> Páginas totales procesadas: 149
>> Total de chunks generados: 500
>> Creando embeddings numéricos e indexando en FAISS...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

>> Descargando y configurando LLM local en GPU...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

>> [ÉXITO] Pipeline RAG inicializado por completo.


In [ ]:
# 3. INTERFAZ INTERACTIVA DE CONSULTA
# Aquí por medio de un while se crea un ciclo para poder realizar las preguntas necesarias, cuando se ejecuta la celda aparece un recuadro para preguntar ya que responde vuelve a aparecer el recuadro hasta que se ponga "salir"
# Hice algunas preguntas, sin embargo se pueden hacer más.
print("¡Asistente RAG Activo! Escribe tus dudas (o escribe 'salir' para finalizar):")

while True:
    pregunta = input("\n Consulta: ")
    if pregunta.lower().strip() == 'salir':
        print(" Ejecución concluida.")
        break
    if not pregunta.strip():
        continue

    print("Extrayendo contexto y generando respuesta...")
    try:
        respuesta = rag_pipeline.invoke(pregunta)
        print(f"\n================ RESPUESTA FUNDAMENTADA ================\n{respuesta}")
        print("========================================================")
    except Exception as e:
        print(f"Error: {e}")

¡Asistente RAG Activo! Escribe tus dudas (o escribe 'salir' para finalizar):

 Consulta: ¿Qué beneficios tiene la inteligencia artificial?


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Extrayendo contexto y generando respuesta...

================ RESPUESTA FUNDAMENTADA ================
 

La inteligencia artificial ha demostrado ser muy útil en muchas áreas, desde la medicina hasta la educación. En la industria automotriz, por ejemplo, la IA puede ayudar a mejorar la eficiencia operativa y reducir costos. Además, en el campo de la robótica, la IA está cambiando cómo trabajamos y interactuamos con nuestros dispositivos.

En términos de salud, la IA puede ayudar a predecir enfermedades antes de que se desarrollen, lo que puede resultar en tratamientos más efectivos y menos invasivos. También puede ser útil en la detección temprana de cáncer y otras condiciones médicas.

En cuanto al empleo, la IA puede reemplazar tareas repetitivas y menores, pero también puede crear nuevas oportunidades de trabajo. Por ejemplo, la IA puede automatizar procesos administrativos, liberando tiempo para especialistas en otros campos.

Además, la IA puede mejorar la seguridad en la industr

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Extrayendo contexto y generando respuesta...

================ RESPUESTA FUNDAMENTADA ================
? La IA podría mejorar la eficiencia y precisión en el diagnóstico, pero también podría reducir la necesidad de humanos en ciertos procesos, lo que podría afectar negativamente la interacción emocional entre el profesional y el paciente. Además, si la IA se encarga de tomar decisiones médicas, esto podría cambiar la naturaleza de la relación entre el médico y el paciente, ya que la confianza y la empatía pueden ser elementos clave en este tipo de relaciones. 

Sin embargo, es importante notar que aunque la IA pueda mejorar la eficiencia y precisión en el diagnóstico, la experiencia humana y emocional sigue siendo fundamental en la atención médica. El doctor debe entender y respetar las emociones de sus pacientes, y la IA solo puede ayudarlo a realizar funciones específicas. De hecho, la IA puede ser una herramienta valiosa para facilitar estos procesos y mejorar la calidad del servici

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Extrayendo contexto y generando respuesta...

================ RESPUESTA FUNDAMENTADA ================
 ¿Cómo puede la empresa garantizar que la utilización de la IA en marketing sea ética y transparente?

La IA tiene un papel crucial en el marketing, pero también presenta riesgos potenciales. Por ejemplo, si la IA no está diseñada correctamente, puede producir resultados inesperados o incluso maliciosos. Además, la IA requiere grandes cantidades de datos para funcionar eficientemente, lo que podría llevar a la pérdida de privacidad de los usuarios.

Para mitigar estos riesgos, la empresa debe asegurarse de que la IA esté diseñada de manera ética y transparente. Esto implica:

1. **Diseño ético**: El diseño de la IA debe ser consciente de los valores y principios éticos de la organización. Esto incluiría considerar aspectos como la privacidad, la seguridad, la inclusión y la diversidad.

2. **Transparencia**: La empresa debe explicar cómo se utiliza la IA y qué tipo de datos se recopil

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Extrayendo contexto y generando respuesta...

================ RESPUESTA FUNDAMENTADA ================
 ¿Y cómo podemos evaluarla?

La inteligencia artificial es una herramienta poderosa que puede ser usada de manera positiva o negativa. Sin embargo, hay que tener cuidado con las implicaciones éticas y sociales de esta tecnología. Es importante entender que la IA tiene potencialidades y riesgos, pero también oportunidades.

En términos de evaluación, la IA se clasifica según su capacidad para aprender y adaptarse a nuevas situaciones. Las métricas comunes son precisión, recall, fórmula de F y error de tipo I y II. Además, se pueden medir otros factores como la eficiencia, la velocidad y la seguridad.

El uso de la IA debe ser consciente y responsable. Se deben establecer límites claros sobre quién controla la IA y cómo se utiliza. También es crucial garantizar que la IA cumpla con los principios éticos y legales.

Finalmente, es importante recordar que la IA no es un sustituto de la in

# **Punto 4: Investigación sobre la Mejora de Rendimiento en Sistemas RAG**

Para transformar un prototipo RAG básico en un sistema de producción eficiente y preciso, se implementan técnicas avanzadas enfocadas en resolver las debilidades de la búsqueda semántica estándar:

### **1. Re-ranking con Modelos de Similitud (Cross-Encoders)**

En un sistema RAG tradicional, la búsqueda inicial se realiza mediante *Bi-Encoders* (como los embeddings de OpenAI o Hugging Face). Estos convierten las frases en vectores independientes y calculan la similitud de coseno. Aunque son extremadamente veloces, sacrifican precisión porque no analizan la interacción palabra por palabra entre la pregunta y el fragmento extraído.
El *Re-ranking* introduce una segunda etapa utilizando un *Cross-Encoder* (como Cohere Rerank o BGE-Reranker). A diferencia del Bi-Encoder, el Cross-Encoder recibe la pregunta del usuario y el fragmento de texto juntos en el mismo bloque de procesamiento. El modelo aplica un mecanismo de atención total para calcular una puntuación de relevancia exacta entre ambos elementos. Al final, reorganiza los fragmentos recuperados en la primera fase y envía al LLM únicamente los k documentos que obtuvieron la calificación real más alta.
#### Pros
 * *Alta Precisión Semántica:* Corrige falsos positivos de la búsqueda vectorial tradicional al capturar dependencias gramaticales profundas.
 * *Mitiga el problema "Lost in the Middle":* Al reducir y reordenar los fragmentos para entregarle al LLM solo lo indispensable, evita que el modelo de lenguaje ignore información crucial que queda sepultada en medio de un prompt demasiado largo.
 * *Permite ampliar la búsqueda inicial:* Puedes configurar el retriever inicial para buscar un volumen alto de documentos (ej. k=25) sin temor a saturar al LLM, ya que el Re-ranker los filtrará eficientemente a un número óptimo (ej. k=3).
#### Contras
 * *Latencia computacional adicionada:* Los Cross-Encoders requieren mucha más potencia de procesamiento por par analizado. Esto puede incrementar el tiempo de respuesta en entornos de producción si se evalúan demasiados fragmentos.
 * *Costo económico:* Depender de APIs externas especializadas en Re-ranking eleva los costos operativos por cada token procesado.

### **2. Segmentación Jerárquica (Parent-Child Retriever)**

Un problema del diseño de un RAG clásico es el tamaño del fragmento. Los fragmentos pequeños (ej. 200 caracteres) capturan embeddings muy específicos y precisos, pero el LLM pierde el contexto macro de la lectura. Fragmentos grandes (ej. 2000 caracteres) retienen el contexto general, pero diluyen el embedding con información secundaria causando ruido.
El método *Parent-Child Retriever* resuelve esto dividiendo el almacenamiento en dos capas conectadas de forma jerárquica:
 1. *Documentos Padres:* El archivo original se corta en fragmentos medianos/grandes para retener el flujo conceptual completo de la sección.
 2. *Documentos Hijos:* Cada fragmento "Padre" se subdivide a su vez en múltiples trozos pequeños, los cuales son los únicos que se convierten en vectores (embeddings) y se guardan en la base de datos como FAISS.
Cuando el usuario hace una pregunta, el sistema busca los vectores de los *trozos hijos* por velocidad y exactitud conceptual. Sin embargo, en lugar de pasarle esos micro-trozos al LLM, el sistema realiza una sustitución en memoria y le entrega al modelo de lenguaje el fragmento *Padre completo* asociado a ese hijo.
#### Pros
 * Garantiza una recuperación rápida basada en términos muy específicos, mientras que provee al LLM un contexto amplio para redactar la respuesta.
 * Evita que el modelo de lenguaje reciba oraciones cortadas a la mitad debido al límite de caracteres del splitter tradicional.
#### Contras
 * Requiere implementar una base de datos relacional adicional o un sistema de mapeo (Key-Value) para enlazar dinámicamente cada ID de los hijos con sus respectivos padres.
 * Al enviar bloques padres enteros, el prompt se vuelve más grande rápidamente, lo que incrementa el costo de tokens si usas APIs de pago.

### **3. Búsqueda Híbrida (Hybrid Search: Dense Embeddings + BM25)**

Los embeddings vectoriales (Dense Retrieval) se destacan interpretando conceptos abstractos, sinónimos e intenciones semánticas, pero tienen una debilidad crítica: fallan en la coincidencia exacta de códigos alfanuméricos, términos técnicos específicos, identificadores de bases de datos o nombres propios raros.
La *Búsqueda Híbrida* combina de forma simultánea dos formas de recuperación:
 1. *Búsqueda Densa:* El enfoque semántico moderno basado en vectores (como FAISS) para comprender de qué trata conceptualmente la pregunta.
 2. *Búsqueda Discreta (Sparse Retrieval):* El enfoque léxico tradicional usando el algoritmo estadístico *BM25* (una versión avanzada de TF-IDF), el cual rastrea coincidencias exactas de palabras clave basadas en su frecuencia de aparición dentro de los documentos.
Cuando se realiza una consulta, ambos algoritmos devuelven su propia lista de documentos candidatos con puntajes basados en escalas diferentes. Para fusionarlas con precisión, se aplica el algoritmo *Reciprocal Rank Fusion (RRF)*, el cual recalcula el peso de cada fragmento según su posición relativa en ambas listas, enviando al LLM una selección unificada de la máxima calidad posible.
#### Pros
 * *Robustez:* Funciona a la perfección tanto para preguntas conceptuales complejas como para consultas que dependen estrictamente de un número de artículo, año o término técnico exacto.
 * *Manejo de lenguaje específico:* Si el usuario emplea palabras clave corporativas o jergas técnicas muy específicas que el modelo de embeddings nunca vio en su entrenamiento, BM25 las rescatará .
#### Contras
 * *Mantenimiento del ecosistema:* Exige configurar, indexar y mantener actualizados dos índices de búsqueda paralelos (la base vectorial y el motor BM25), duplicando los requerimientos de almacenamiento en memoria.
 * *Ajuste de hiperparámetros complejo:* Encontrar el balance perfecto de peso para el algoritmo RRF (qué tanto priorizar la búsqueda semántica frente a las palabras clave exactas) requiere de un proceso constante de experimentación según la naturaleza de tus datos.


## **Referencias**

 1. *Sobre Re-ranking (Cross-Encoders):*
   * Thakur, N., Reimers, N., Daxenberger, J., & Gurevych, I. (2021). BEIR: A Heterogeneous Benchmark for Systematic Evaluation of Information Retrieval Models. arXiv preprint arXiv:2104.08663.
   * Explicación del uso práctico: Reimers & Gurevych (2019) introdujeron Sentence-BERT, donde detallan formalmente el costo y precisión de contrastar arquitecturas Bi-Encoder vs Cross-Encoder.
 2. *Sobre Segmentación Jerárquica y Limitaciones de Contexto:*
   * Liu, N. F., Lin, K., Hewitt, J., Paranjape, A., Bevilacqua, M., Petroni, F., & Liang, P. (2024). Lost in the Middle: How Language Models Use Long Contexts. Transactions of the Association for Computational Linguistics, 12, 157-173.
   * Explicación del uso práctico: Documentación oficial de LangChain (2024) sobre los patrones de diseño de arquitecturas empresariales avanzados bajo el estándar Parent-Document Retriever.
 3. *Sobre Búsqueda Híbrida y Fusión RRF:*
   * Cormack, G. V., Clarke, C. L., & Buettcher, S. (2009). Reciprocal rank fusion outperforms elegance in retrieval. Proceedings of the 32nd international ACM SIGIR conference on Research and development in information retrieval, 105-112.
   * Robertson, S., & Zaragoza, H. (2009). The Probabilistic Relevance Framework: BM25 and Beyond. Foundations and Trends in Information Retrieval, 3(4), 333-380.